# 4. LangChain + Model Context Protocol (MCP)
Advanced architecture combining **LangChain agents** with **MCP tool servers**.

In [ ]:
!pip install -q langchain langchain-openai

In [ ]:
# Simulated MCP server

class MCPServer:

    def __init__(self):
        self.tools = {}

    def register(self, name, func):
        self.tools[name] = func

    def call(self, name, arg):
        return self.tools[name](arg)


def stock_price(company):
    data = {"apple": 190, "google": 140}
    return data.get(company.lower(), "unknown")

server = MCPServer()
server.register("stock_price", stock_price)

server.tools

In [ ]:
# Wrap MCP tools as LangChain tools

from langchain.tools import Tool

def stock_tool(company):
    return server.call("stock_price", company)

tools = [
    Tool(
        name="Stock Price Tool",
        func=stock_tool,
        description="Get stock price for a company"
    )
]

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent

llm = ChatOpenAI(model="gpt-4o-mini")

agent = initialize_agent(
    tools,
    llm,
    agent="zero-shot-react-description",
    verbose=True
)

agent.run("What is the stock price of Apple?")

In [ ]:
# Agent middleware example

def logging(agent_func):

    def wrapper(query):
        print("Agent received:", query)
        result = agent_func(query)
        print("Agent output:", result)
        return result

    return wrapper

safe_agent = logging(agent.run)

safe_agent("What is the stock price of Google?")